# LeetCode 1700 — Number of Students Unable to Eat Lunch bằng Python

**Link đề:** https://leetcode.com/problems/number-of-students-unable-to-eat-lunch/
**Độ khó:** Easy · **Chủ đề:** Array, Stack, Queue, Simulation

```python
class Solution:
    def countStudents(self, students: List[int], sandwiches: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Phân biệt **hàng đợi (queue, FIFO)** và **ngăn xếp (stack, LIFO)**; biết vì sao `list.pop(0)` chậm và dùng `collections.deque`.
2. Mô phỏng đúng đề, và quan trọng nhất: **tìm điều kiện dừng** để không lặp vô hạn.
3. Phát hiện insight: **thứ tự học sinh trong hàng không quan trọng**, chỉ cần **đếm** số người thích loại 0 và loại 1.
4. Trace mô phỏng và trace cách đếm; chứng minh vì sao cách đếm đúng.
5. Nhận diện pattern "mô phỏng xoay vòng → thay bằng đếm".

## 1. Đề bài và ví dụ

Căng-tin có bánh **tròn (0)** và **vuông (1)**. Học sinh xếp **hàng đợi**; bánh xếp thành **chồng (stack)**, số bánh = số học sinh. Mỗi bước:
- Nếu học sinh **đầu hàng** thích đúng loại bánh **trên cùng chồng** → lấy bánh và rời hàng.
- Nếu không → bỏ qua, quay về **cuối hàng**.

Quá trình dừng khi **không học sinh nào trong hàng** muốn bánh trên cùng. Trả về số học sinh **không được ăn**.

`students[j]` là sở thích của người thứ `j` (j = 0 là đầu hàng); `sandwiches[i]` là loại bánh thứ `i` (i = 0 là **trên cùng**).

| `students` | `sandwiches` | Kết quả |
|---|---|:---:|
| `[1,1,0,0]` | `[0,1,0,1]` | `0` |
| `[1,1,1,0,0,1]` | `[1,0,0,0,1,1]` | `3` |

**Constraints:** `1 <= len(students) == len(sandwiches) <= 100`; mọi phần tử là `0` hoặc `1`.

### 1.1. Đọc ví dụ 2 bằng tay

Hàng `[1,1,1,0,0,1]` (4 người thích 1, 2 người thích 0), chồng bánh `[1,0,0,0,1,1]`.
- Bánh trên cùng `1` → người đầu thích 1, lấy. Hàng `[1,1,0,0,1]`, bánh `[0,0,0,1,1]`.
- Bánh `0`: người `1` → cuối hàng, người `1` → cuối, người `0` lấy... tiếp tục, 2 người thích 0 lần lượt ăn 2 bánh `0`.
- Còn bánh `0` thứ ba trên cùng, nhưng hàng chỉ còn 3 người thích `1` → **kẹt**. Đáp án `3`.

In [1]:
from typing import List

students = [1, 1, 1, 0, 0, 1]
sandwiches = [1, 0, 0, 0, 1, 1]
print("số người thích 0:", students.count(0), "| thích 1:", students.count(1))
print("số bánh 0:", sandwiches.count(0), "| bánh 1:", sandwiches.count(1))

số người thích 0: 2 | thích 1: 4
số bánh 0: 3 | bánh 1: 3


## 2. Nền tảng Python cần biết

### 2.1. Queue (FIFO) và Stack (LIFO)

- **Queue (hàng đợi):** vào cuối, ra đầu — *First In First Out*. Học sinh xếp hàng.
- **Stack (ngăn xếp):** vào/ra cùng một đầu — *Last In First Out*. Chồng bánh: chỉ lấy được cái trên cùng.

Ở bài này, chồng bánh **không thêm bánh mới**, chỉ lấy từ trên xuống ⇒ duyệt `sandwiches` từ index 0 tăng dần là đủ.

In [2]:
# Stack bằng list: append / pop ở cuối
stack = []
stack.append("A"); stack.append("B"); stack.append("C")
print("stack:", stack, "-> pop:", stack.pop(), "-> còn:", stack)

# Queue "ngây thơ" bằng list: append ở cuối, pop(0) ở đầu
queue = ["An", "Bình", "Chi"]
queue.append("Dũng")
print("queue:", queue, "-> ra đầu:", queue.pop(0), "-> còn:", queue)

stack: ['A', 'B'] -> pop: C -> còn: ['A', 'B']
queue: ['Bình', 'Chi', 'Dũng'] -> ra đầu: An -> còn: ['Bình', 'Chi', 'Dũng']


### 2.2. Vì sao `list.pop(0)` chậm? — `collections.deque`

`list.pop(0)` phải **dịch tất cả phần tử còn lại** sang trái một ô → `O(n)`. `collections.deque` (double-ended queue) lấy/thêm ở **cả hai đầu** trong `O(1)`: `append`, `appendleft`, `pop`, `popleft`. Ngoài ra `rotate(-1)` đưa phần tử đầu xuống cuối.

In [3]:
from collections import deque

q = deque([1, 1, 0, 0])
print("ban đầu  :", q)
front = q.popleft()          # lấy người đầu hàng
q.append(front)              # đưa về cuối hàng
print("đưa đầu xuống cuối:", q)
q.rotate(-1)                 # cách viết khác cho thao tác trên
print("rotate(-1):", q)
print("q[0] (đầu hàng) =", q[0], "| len =", len(q))

ban đầu  : deque([1, 1, 0, 0])
đưa đầu xuống cuối: deque([1, 0, 0, 1])
rotate(-1): deque([0, 0, 1, 1])
q[0] (đầu hàng) = 0 | len = 4


In [4]:
import time
N = 50_000
lst = list(range(N)); t = time.perf_counter()
while lst: lst.pop(0)
t_list = time.perf_counter() - t
dq = deque(range(N)); t = time.perf_counter()
while dq: dq.popleft()
t_dq = time.perf_counter() - t
print(f"list.pop(0): {t_list:.4f}s | deque.popleft(): {t_dq:.4f}s")

list.pop(0): 0.2217s | deque.popleft(): 0.0032s


### 2.3. Đếm: `list.count`, `collections.Counter`, `sum`

Vì phần tử chỉ là 0/1: `sum(students)` = số người thích 1; `len - sum` = số người thích 0. `Counter` đếm tổng quát cho mọi giá trị.

In [5]:
from collections import Counter

students = [1, 1, 0, 0, 1]
print("count(0) =", students.count(0), "| count(1) =", students.count(1))
print("sum =", sum(students), "-> thích 1; len - sum =", len(students) - sum(students), "-> thích 0")
cnt = Counter(students)
print(cnt, "| cnt[0] =", cnt[0], "| cnt[7] =", cnt[7], "(key không có -> 0)")

count(0) = 2 | count(1) = 3
sum = 3 -> thích 1; len - sum = 2 -> thích 0
Counter({1: 3, 0: 2}) | cnt[0] = 2 | cnt[7] = 0 (key không có -> 0)


### 2.4. Mảng đếm 2 ô `cnt = [c0, c1]`

Giá trị chính là chỉ số: `cnt[s] += 1`. Gọn hơn `Counter` khi chỉ có giá trị 0/1.

In [6]:
cnt = [0, 0]
for s in [1, 1, 1, 0, 0, 1]:
    cnt[s] += 1
print("cnt =", cnt, "-> 2 người thích 0, 4 người thích 1")

cnt = [2, 4] -> 2 người thích 0, 4 người thích 1


## 3. Cách trực giác — mô phỏng đúng như đề

Khó nhất là **điều kiện dừng**: hàng có thể xoay vòng mãi. Ta dừng khi đã **xoay hết một vòng** (đếm `skips` liên tiếp = độ dài hàng) mà không ai lấy bánh.

- `skips` = số lần liên tiếp học sinh đầu hàng **từ chối**.
- Ai đó lấy bánh → reset `skips = 0`.
- Nếu `skips == len(queue)` → mọi người đều đã nhìn thấy bánh trên cùng và từ chối → dừng.

In [7]:
def count_students_simulation(students: List[int], sandwiches: List[int]) -> int:
    queue = deque(students)
    top = 0                         # chỉ số bánh trên cùng
    skips = 0
    while queue and skips < len(queue):
        if queue[0] == sandwiches[top]:
            queue.popleft()         # lấy bánh, rời hàng
            top += 1
            skips = 0
        else:
            queue.rotate(-1)        # về cuối hàng
            skips += 1
    return len(queue)

print(count_students_simulation([1, 1, 0, 0], [0, 1, 0, 1]))              # 0
print(count_students_simulation([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))  # 3

0
3


### Chi phí của mô phỏng

Mỗi bánh được lấy sau tối đa `len(queue)` lần xoay ⇒ worst case `O(n²)` bước (`n ≤ 100` → ≤ 10.000 bước, ổn). Nhưng ta đang làm nhiều việc vô ích: xoay hàng chỉ để tìm **ai đó** thích bánh trên cùng.

## 4. Insight — thứ tự hàng không quan trọng

Hỏi: bánh trên cùng loại `t` có được ai lấy không?
- Nếu trong hàng **còn ít nhất một** người thích `t`, hàng sẽ xoay cho tới khi người đó lên đầu và lấy. **Ai** lấy không quan trọng — chỉ cần biết **còn hay hết**.
- Nếu **không còn ai** thích `t` → kẹt vĩnh viễn; mọi người còn lại đều không ăn được.

⇒ Chỉ cần đếm `cnt[0]`, `cnt[1]`. Duyệt bánh từ trên xuống: nếu `cnt[t] > 0` thì `cnt[t] -= 1`, ngược lại dừng; đáp án là `cnt[0] + cnt[1]` lúc dừng.

In [8]:
# Minh hoạ: hai hàng cùng số lượng 0/1 nhưng khác thứ tự cho cùng đáp án
sw = [1, 0, 0, 0, 1, 1]
for st in ([1, 1, 1, 0, 0, 1], [0, 1, 1, 1, 1, 0], [1, 0, 1, 0, 1, 1]):
    print(st, "->", count_students_simulation(st, sw))

[1, 1, 1, 0, 0, 1] -> 3
[0, 1, 1, 1, 1, 0] -> 3
[1, 0, 1, 0, 1, 1] -> 3


## 5. Lời giải chính — đếm

**Đoạn 1 — đếm sở thích:** `cnt = [0, 0]`; với mỗi `s` trong `students`, `cnt[s] += 1`.

**Đoạn 2 — duyệt bánh từ trên xuống:** với mỗi `t` trong `sandwiches`:
- `cnt[t] == 0` → không ai muốn, **dừng** (`break`).
- ngược lại `cnt[t] -= 1` (một người thích `t` đã ăn).

**Đoạn 3 — trả về** `cnt[0] + cnt[1]`.

In [9]:
def count_students_counting(students: List[int], sandwiches: List[int]) -> int:
    cnt = [0, 0]
    for s in students:              # đoạn 1
        cnt[s] += 1
    for t in sandwiches:            # đoạn 2
        if cnt[t] == 0:
            break
        cnt[t] -= 1
    return cnt[0] + cnt[1]          # đoạn 3

print(count_students_counting([1, 1, 0, 0], [0, 1, 0, 1]))              # 0
print(count_students_counting([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))  # 3

0
3


## 6. TRACE từng bước

### 6.1. Trace cách đếm — ví dụ 2

Ban đầu `cnt = [2, 4]` (2 người thích 0, 4 người thích 1).

| i | bánh `t` | cnt trước | còn ai thích `t`? | cnt sau |
|---:|:---:|---|---|---|
| 0 | 1 | [2, 4] | có | [2, 3] |
| 1 | 0 | [2, 3] | có | [1, 3] |
| 2 | 0 | [1, 3] | có | [0, 3] |
| 3 | 0 | [0, 3] | **không** → dừng | [0, 3] |

Đáp án `0 + 3 = 3`.

In [10]:
def trace_counting(students, sandwiches):
    cnt = [students.count(0), students.count(1)]
    print("cnt ban đầu =", cnt)
    for i, t in enumerate(sandwiches):
        if cnt[t] == 0:
            print(f"i={i}: bánh {t}, cnt={cnt} -> không còn ai thích {t} -> DỪNG")
            break
        before = cnt.copy()
        cnt[t] -= 1
        print(f"i={i}: bánh {t}, cnt {before} -> {cnt}")
    else:
        print("Phát hết bánh!")
    print("Không ăn được:", sum(cnt))
    return sum(cnt)

trace_counting([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1])
print()
trace_counting([1, 1, 0, 0], [0, 1, 0, 1])

cnt ban đầu = [2, 4]
i=0: bánh 1, cnt [2, 4] -> [2, 3]
i=1: bánh 0, cnt [2, 3] -> [1, 3]
i=2: bánh 0, cnt [1, 3] -> [0, 3]
i=3: bánh 0, cnt=[0, 3] -> không còn ai thích 0 -> DỪNG
Không ăn được: 3

cnt ban đầu = [2, 2]
i=0: bánh 0, cnt [2, 2] -> [1, 2]
i=1: bánh 1, cnt [1, 2] -> [1, 1]
i=2: bánh 0, cnt [1, 1] -> [0, 1]
i=3: bánh 1, cnt [0, 1] -> [0, 0]
Phát hết bánh!
Không ăn được: 0


0

**Cú pháp mới — `for ... else`:** khối `else` của vòng `for` chạy **chỉ khi vòng lặp kết thúc bình thường** (không gặp `break`). Ở trên nó in "Phát hết bánh!" khi mọi bánh đều có người lấy.

### 6.2. Trace mô phỏng hàng đợi — ví dụ 1 (giống giải thích của đề)

In [11]:
def trace_simulation(students, sandwiches):
    queue = deque(students)
    top, skips, step = 0, 0, 0
    while queue and skips < len(queue):
        step += 1
        if queue[0] == sandwiches[top]:
            queue.popleft(); top += 1; skips = 0
            action = "LẤY bánh"
        else:
            queue.rotate(-1); skips += 1
            action = "về cuối  "
        print(f"bước {step:>2}: {action} | hàng = {list(queue)} | bánh còn = {sandwiches[top:]} | skips = {skips}")
    print("Không ăn được:", len(queue))
    return len(queue)

trace_simulation([1, 1, 0, 0], [0, 1, 0, 1])

bước  1: về cuối   | hàng = [1, 0, 0, 1] | bánh còn = [0, 1, 0, 1] | skips = 1
bước  2: về cuối   | hàng = [0, 0, 1, 1] | bánh còn = [0, 1, 0, 1] | skips = 2
bước  3: LẤY bánh | hàng = [0, 1, 1] | bánh còn = [1, 0, 1] | skips = 0
bước  4: về cuối   | hàng = [1, 1, 0] | bánh còn = [1, 0, 1] | skips = 1
bước  5: LẤY bánh | hàng = [1, 0] | bánh còn = [0, 1] | skips = 0
bước  6: về cuối   | hàng = [0, 1] | bánh còn = [0, 1] | skips = 1
bước  7: LẤY bánh | hàng = [1] | bánh còn = [1] | skips = 0
bước  8: LẤY bánh | hàng = [] | bánh còn = [] | skips = 0
Không ăn được: 0


0

In [12]:
trace_simulation([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1])

bước  1: LẤY bánh | hàng = [1, 1, 0, 0, 1] | bánh còn = [0, 0, 0, 1, 1] | skips = 0
bước  2: về cuối   | hàng = [1, 0, 0, 1, 1] | bánh còn = [0, 0, 0, 1, 1] | skips = 1
bước  3: về cuối   | hàng = [0, 0, 1, 1, 1] | bánh còn = [0, 0, 0, 1, 1] | skips = 2
bước  4: LẤY bánh | hàng = [0, 1, 1, 1] | bánh còn = [0, 0, 1, 1] | skips = 0
bước  5: LẤY bánh | hàng = [1, 1, 1] | bánh còn = [0, 1, 1] | skips = 0
bước  6: về cuối   | hàng = [1, 1, 1] | bánh còn = [0, 1, 1] | skips = 1
bước  7: về cuối   | hàng = [1, 1, 1] | bánh còn = [0, 1, 1] | skips = 2
bước  8: về cuối   | hàng = [1, 1, 1] | bánh còn = [0, 1, 1] | skips = 3
Không ăn được: 3


3

Ở cuối trace thứ hai: hàng `[1, 1, 1]`, bánh trên cùng `0`, `skips` tăng tới 3 = độ dài hàng → dừng. Đúng như cách đếm dự đoán.

## 7. Các lời giải khác

### 7.1. Mô phỏng bằng list + `pop(0)` (cách "ngây thơ" nhất)

Giống mô phỏng ở mục 3 nhưng dùng list. Đúng nhưng mỗi `pop(0)` tốn `O(n)`.

In [13]:
def count_students_list(students: List[int], sandwiches: List[int]) -> int:
    queue = students[:]            # sao chép để không sửa input
    stack = sandwiches[:]
    skips = 0
    while queue and skips < len(queue):
        s = queue.pop(0)
        if s == stack[0]:
            stack.pop(0)
            skips = 0
        else:
            queue.append(s)
            skips += 1
    return len(queue)

print(count_students_list([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))

3


### 7.2. Mô phỏng với điều kiện dừng "không còn ai thích bánh trên cùng"

Kết hợp hai ý: vẫn mô phỏng hàng đợi, nhưng kiểm tra dừng bằng `t not in queue` (thay vì đếm `skips`). Dễ hiểu nhưng `in` trên deque tốn `O(n)`.

In [14]:
def count_students_membership(students: List[int], sandwiches: List[int]) -> int:
    queue = deque(students)
    for t in sandwiches:
        if t not in queue:          # không ai muốn bánh này -> kẹt
            break
        while queue[0] != t:        # xoay tới người thích t
            queue.rotate(-1)
        queue.popleft()
    return len(queue)

print(count_students_membership([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))

3


### 7.3. Đếm bằng `sum` (chỉ vì giá trị là 0/1)

`ones = sum(students)`, `zeros = n - ones`. Khi gặp bánh không ai thích, số người còn lại = `n - i` (`i` là số bánh đã phát). Không cần cộng `cnt[0] + cnt[1]`.

In [15]:
def count_students_sum(students: List[int], sandwiches: List[int]) -> int:
    n = len(students)
    ones = sum(students)
    zeros = n - ones
    for i, t in enumerate(sandwiches):
        if t == 1:
            if ones == 0:
                return n - i
            ones -= 1
        else:
            if zeros == 0:
                return n - i
            zeros -= 1
    return 0

print(count_students_sum([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]), count_students_sum([1, 1, 0, 0], [0, 1, 0, 1]))

3 0


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Mô phỏng list `pop(0)` | đúng nguyên văn đề | `O(n³)` worst | `O(n)` | chậm nhất (pop(0) là O(n)) |
| Mô phỏng deque + skips | xoay hàng, dừng khi xoay hết vòng | `O(n²)` | `O(n)` | điều kiện dừng là then chốt |
| Mô phỏng + `t not in queue` | kiểm tra trước khi xoay | `O(n²)` | `O(n)` | dễ hiểu |
| **Đếm `cnt[0], cnt[1]`** | thứ tự hàng không quan trọng | `O(n)` | `O(1)` | khuyên dùng |
| Đếm bằng `sum` | biến thể của đếm | `O(n)` | `O(1)` | trả `n - i` |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def countStudents(self, students: List[int], sandwiches: List[int]) -> int:
        cnt = [0, 0]
        for s in students:
            cnt[s] += 1
        for t in sandwiches:
            if cnt[t] == 0:
                break
            cnt[t] -= 1
        return cnt[0] + cnt[1]

sol = Solution()
print(sol.countStudents([1, 1, 0, 0], [0, 1, 0, 1]), sol.countStudents([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))

0 3


Bản mô phỏng bằng `deque` (nếu muốn nộp đúng tinh thần "Queue/Simulation"):

In [17]:
class SolutionSimulation:
    def countStudents(self, students: List[int], sandwiches: List[int]) -> int:
        q = deque(students)
        top = skips = 0
        while q and skips < len(q):
            if q[0] == sandwiches[top]:
                q.popleft(); top += 1; skips = 0
            else:
                q.rotate(-1); skips += 1
        return len(q)

print(SolutionSimulation().countStudents([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))

3


## 9. Vì sao đúng?

**Bổ đề (thứ tự không quan trọng):** Xét bánh trên cùng loại `t`.
- Nếu hàng còn người thích `t`: hàng chỉ xoay (không ai rời đi) cho tới khi người thích `t` **đầu tiên** lên đầu và lấy bánh. Sau đó số người thích `t` giảm 1, số người thích `1 - t` giữ nguyên. Thứ tự hàng mới có thể khác, nhưng **số lượng** mỗi loại chỉ phụ thuộc vào số lượng trước đó.
- Nếu không ai thích `t`: mọi người đều từ chối, bánh `t` không bao giờ được lấy, và vì chỉ lấy được bánh trên cùng nên **mọi bánh bên dưới cũng không**. Quá trình dừng.

**Invariant của cách đếm:** trước khi xét bánh thứ `i`, `cnt[0]`, `cnt[1]` đúng bằng số người thích 0/1 còn trong hàng của mô phỏng thật. Bổ đề cho thấy mỗi bước cập nhật giữ invariant; khi dừng (hết bánh hoặc kẹt), `cnt[0] + cnt[1]` là số người còn lại = số người không được ăn.

**Điều kiện dừng của mô phỏng `skips`:** nếu `skips == len(queue)`, mỗi người trong hàng đã đứng đầu đúng một lần và từ chối cùng một bánh trên cùng (bánh không đổi khi không ai lấy) ⇒ không ai thích `t` ⇒ kẹt vĩnh viễn.

## 10. Độ phức tạp

Gọi `n = len(students) = len(sandwiches)`.
- **Đếm:** đếm `O(n)` + duyệt bánh `O(n)` ⇒ **Time `O(n)`**, **Space `O(1)`** (mảng 2 ô).
- **Mô phỏng deque:** mỗi bánh cần tối đa `n` lần xoay ⇒ `O(n²)`, Space `O(n)`.
- **Mô phỏng list `pop(0)`:** thêm hệ số `O(n)` cho mỗi `pop(0)` ⇒ `O(n³)` worst case.

In [18]:
import random

def count_steps(students, sandwiches):
    q, top, skips, steps = deque(students), 0, 0, 0
    while q and skips < len(q):
        steps += 1
        if q[0] == sandwiches[top]:
            q.popleft(); top += 1; skips = 0
        else:
            q.rotate(-1); skips += 1
    return steps

for n in [10, 100, 1000]:
    st = [0] * (n // 2) + [1] * (n // 2)       # trường hợp xấu: người thích 1 đứng trước
    sw = [1, 0] * (n // 2)
    print(f"n = {n:>4}: số bước mô phỏng = {count_steps(st, sw):>7} | số bước đếm ≈ {2 * n}")

n =   10: số bước mô phỏng =      35 | số bước đếm ≈ 20
n =  100: số bước mô phỏng =    2600 | số bước đếm ≈ 200
n = 1000: số bước mô phỏng =  251000 | số bước đếm ≈ 2000


## 11. Bộ test

1. Hai ví dụ đề.
2. Biên: 1 học sinh ăn được / không ăn được.
3. Dễ sai: mọi người cùng thích một loại nhưng bánh trên cùng loại kia (kẹt ngay, đáp án `n`); kẹt ở bánh **cuối cùng**; phát hết.
4. Đối chiếu 6 cài đặt trên 2000 bộ ngẫu nhiên (cùng độ dài).

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1", S.countStudents([1, 1, 0, 0], [0, 1, 0, 1]), 0)
check("đề 2", S.countStudents([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]), 3)
check("1 người ăn được", S.countStudents([0], [0]), 0)
check("1 người không ăn", S.countStudents([1], [0]), 1)
check("kẹt ngay từ đầu", S.countStudents([1, 1, 1], [0, 1, 1]), 3)
check("kẹt ở bánh cuối", S.countStudents([0, 0, 1], [0, 0, 0]), 1)
check("phát hết", S.countStudents([0, 1, 0, 1], [1, 1, 0, 0]), 0)

PASS đề 1: 0
PASS đề 2: 3
PASS 1 người ăn được: 0
PASS 1 người không ăn: 1
PASS kẹt ngay từ đầu: 3
PASS kẹt ở bánh cuối: 1
PASS phát hết: 0


In [20]:
impls = [count_students_simulation, count_students_counting, count_students_list,
         count_students_membership, count_students_sum,
         Solution().countStudents, SolutionSimulation().countStudents]
random.seed(1700)
for _ in range(2000):
    n = random.randint(1, 12)
    st = [random.randint(0, 1) for _ in range(n)]
    sw = [random.randint(0, 1) for _ in range(n)]
    res = {f(st[:], sw[:]) for f in impls}
    assert len(res) == 1, (st, sw, res)
print("Đối chiếu 7 cài đặt trên 2000 bộ ngẫu nhiên: PASS")

Đối chiếu 7 cài đặt trên 2000 bộ ngẫu nhiên: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Mô phỏng không có điều kiện dừng đúng → lặp vô hạn

Viết `while queue:` và chỉ xoay khi không khớp. Khi kẹt, hàng xoay mãi. (Demo dưới đây có `max_steps` để không treo notebook.)

In [21]:
def wrong_infinite(students, sandwiches, max_steps=1000):
    q, top, steps = deque(students), 0, 0
    while q and steps < max_steps:          # thiếu điều kiện dừng thật sự
        steps += 1
        if q[0] == sandwiches[top]:
            q.popleft(); top += 1
        else:
            q.rotate(-1)
    return steps

print("số bước trước khi bị chặn:", wrong_infinite([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]), "-> đáng lẽ dừng sau 8 bước (xem trace mục 6.2)")

số bước trước khi bị chặn: 1000 -> đáng lẽ dừng sau 8 bước (xem trace mục 6.2)


**Sửa:** thêm biến `skips` (dừng khi `skips == len(queue)`) hoặc kiểm tra `t not in queue`.

### Lỗi 2 — Đếm `min(số người thích t, số bánh t)` cho từng loại

Nghĩ rằng "mỗi loại ghép tối đa min(cung, cầu)" mà quên rằng bánh xếp **chồng**: một bánh kẹt ở trên chặn **mọi** bánh bên dưới.

In [22]:
def wrong_min_match(students, sandwiches):
    eaten = sum(min(students.count(t), sandwiches.count(t)) for t in (0, 1))
    return len(students) - eaten

st, sw = [1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]
print("sai:", wrong_min_match(st, sw), "| đúng:", count_students_counting(st, sw))

sai: 1 | đúng: 3


Ở ví dụ 2, bánh `0` thứ ba chặn hai bánh `1` bên dưới dù vẫn còn người thích `1`. **Sửa:** duyệt bánh **theo thứ tự** và `break` ở bánh đầu tiên không ai muốn.

### Lỗi 3 — Quên `break`, chỉ bỏ qua bánh không ai thích

In [23]:
def wrong_no_break(students, sandwiches):
    cnt = [students.count(0), students.count(1)]
    for t in sandwiches:
        if cnt[t] > 0:
            cnt[t] -= 1          # SAI: bánh không ai thích bị "bỏ qua" thay vì chặn
    return sum(cnt)

print("sai:", wrong_no_break(st, sw), "| đúng:", count_students_counting(st, sw))

sai: 1 | đúng: 3


**Sửa:** gặp `cnt[t] == 0` thì `break` ngay.

### Lỗi 4 — Nhầm chiều chồng bánh

Tưởng `sandwiches[-1]` là trên cùng (vì stack Python thường `pop()` ở cuối). Đề nói rõ **`i = 0` là trên cùng**. **Sửa:** duyệt `sandwiches` từ đầu (hoặc đảo ngược trước khi dùng `pop()`).

## 13. Mẹo nhận dạng pattern

- Mô phỏng có thao tác **xoay vòng** (đưa xuống cuối hàng) → thường thứ tự **không quan trọng**, hãy thử thay bằng **đếm**.
- Giá trị chỉ có vài loại (0/1, A–Z) → mảng đếm kích thước nhỏ.
- Mô phỏng có thể lặp vô hạn → phải xác định rõ **điều kiện dừng** (một vòng đầy đủ không có tiến triển).
- Cần queue trong Python → `collections.deque`, không dùng `list.pop(0)`.
- Bài tương tự: 2073 Time Needed to Buy Tickets (đếm thay mô phỏng), 950 Reveal Cards In Increasing Order (deque), 933 Number of Recent Calls (queue).

## 14. Biến thể

### Biến thể 1 — Có `k` loại bánh

Không thay đổi gì về ý tưởng: dùng `Counter` thay mảng 2 ô.

In [24]:
def unable_k_types(students, sandwiches):
    cnt = Counter(students)
    for t in sandwiches:
        if cnt[t] == 0:
            break
        cnt[t] -= 1
    return sum(cnt.values())

print(unable_k_types([2, 0, 1, 2, 1], [1, 2, 2, 0, 1]))   # phát hết -> 0
print(unable_k_types([2, 2, 1], [1, 0, 2]))               # kẹt ở bánh 0 -> 2

0
2


### Biến thể 2 — Trả về **danh sách** chỉ số học sinh không được ăn

Bây giờ thứ tự **có** quan trọng (cần biết là ai) → phải mô phỏng thật.

In [25]:
def unable_ids(students, sandwiches):
    q = deque(range(len(students)))      # lưu chỉ số thay cho sở thích
    top = skips = 0
    while q and skips < len(q):
        if students[q[0]] == sandwiches[top]:
            q.popleft(); top += 1; skips = 0
        else:
            q.rotate(-1); skips += 1
    return sorted(q)

print(unable_ids([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]))

[1, 2, 5]


### Biến thể 3 — Thời gian mua vé (ý tưởng LeetCode 2073)

Mỗi người mua 1 vé mất 1 giây, ai còn cần vé thì quay về cuối hàng. Hỏi người `k` mua xong lúc nào? Thay mô phỏng bằng đếm: người đứng trước-hoặc-bằng `k` đóng góp `min(t_i, t_k)`, người đứng sau đóng góp `min(t_i, t_k - 1)`.

In [26]:
def time_to_buy(tickets, k):
    return sum(min(t, tickets[k]) if i <= k else min(t, tickets[k] - 1)
               for i, t in enumerate(tickets))

print(time_to_buy([2, 3, 2], 2), time_to_buy([5, 1, 1, 1], 0))   # 6, 8

6 8


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính tay
`students = [0, 1, 1, 1]`, `sandwiches = [1, 1, 0, 1]`. Lập bảng `cnt` theo từng bánh và cho biết đáp án.

In [27]:
def exercise_1() -> int:
    # TODO: trả về đáp án bạn tính tay
    pass

# assert exercise_1() == Solution().countStudents([0, 1, 1, 1], [1, 1, 0, 1])

### Bài 2 — Điền chỗ trống cách đếm

In [28]:
def counting_fill(students, sandwiches):
    cnt = [0, 0]
    # TODO 1: đếm sở thích vào cnt
    # TODO 2: duyệt bánh; nếu cnt[t] == 0 thì dừng, ngược lại giảm
    # TODO 3: trả về tổng còn lại
    pass

# assert counting_fill([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 3

### Bài 3 — Mô phỏng với `t not in queue`
Tự viết lại mô phỏng dùng `deque`, kiểm tra kẹt bằng `t not in queue`.

In [29]:
def simulate_membership(students, sandwiches):
    # TODO
    pass

# assert simulate_membership([1, 1, 0, 0], [0, 1, 0, 1]) == 0

### Bài 4 — Số bánh được phát
Trả về **số bánh được phát** (thay vì số người không ăn).

In [30]:
def sandwiches_served(students, sandwiches):
    # TODO
    pass

# assert sandwiches_served([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 3

### Bài 5 — Biến thể `k` loại, trả về loại bánh gây kẹt
Trả về loại bánh trên cùng khi kẹt, hoặc `None` nếu phát hết.

In [31]:
def blocking_type(students, sandwiches):
    # TODO: dùng Counter
    pass

# assert blocking_type([2, 2, 1], [1, 0, 2]) == 0
# assert blocking_type([0, 1], [1, 0]) is None

### Bài 6 — Mini challenge: sắp lại chồng bánh
Cho `students` và `sandwiches` (cùng tổng số bánh mỗi loại có thể khác số người). Có thể **sắp xếp lại** chồng bánh tuỳ ý. Số người không ăn **nhỏ nhất** là bao nhiêu? Trả về số đó.

In [32]:
def min_unable_rearranged(students, sandwiches):
    # TODO: suy nghĩ đặt bánh nào lên trên trước
    pass

# assert min_unable_rearranged([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 1

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đếm: 1 người thích 0, 3 người thích 1. (2) Duyệt bánh `1, 1, 0, 1`. (3) `[1,3]→[1,2]→[1,1]→[0,1]→[0,0]` → 0.

**Bài 2:** (1) Như mục 5. (2) `cnt[s] += 1`; `if cnt[t] == 0: break`. (3) `return cnt[0] + cnt[1]`.

**Bài 3:** (1) Duyệt `t` trong `sandwiches`. (2) `if t not in q: break`. (3) `while q[0] != t: q.rotate(-1)`; rồi `q.popleft()`.

**Bài 4:** (1) Bánh được phát = người được ăn. (2) `n - số người không ăn`. (3) Hoặc đếm số lần `cnt[t] -= 1`.

**Bài 5:** (1) Như cách đếm. (2) Khi `cnt[t] == 0` thì `return t`. (3) Hết vòng lặp `return None`.

**Bài 6:** (1) Đặt bánh có người thích lên trước. (2) Mỗi loại ghép được `min(cầu, cung)`; bánh thừa để dưới cùng. (3) `n - sum(min(students.count(t), sandwiches.count(t)) for t in (0, 1))`.

## 17. Đáp án tham khảo

In [33]:
def ans_exercise_1():
    return 0

def ans_counting_fill(students, sandwiches):
    cnt = [0, 0]
    for s in students:
        cnt[s] += 1
    for t in sandwiches:
        if cnt[t] == 0:
            break
        cnt[t] -= 1
    return cnt[0] + cnt[1]

def ans_simulate_membership(students, sandwiches):
    q = deque(students)
    for t in sandwiches:
        if t not in q:
            break
        while q[0] != t:
            q.rotate(-1)
        q.popleft()
    return len(q)

def ans_sandwiches_served(students, sandwiches):
    return len(students) - ans_counting_fill(students, sandwiches)

def ans_blocking_type(students, sandwiches):
    cnt = Counter(students)
    for t in sandwiches:
        if cnt[t] == 0:
            return t
        cnt[t] -= 1
    return None

def ans_min_unable_rearranged(students, sandwiches):
    matched = sum(min(students.count(t), sandwiches.count(t)) for t in set(students) | set(sandwiches))
    return len(students) - matched

assert ans_exercise_1() == Solution().countStudents([0, 1, 1, 1], [1, 1, 0, 1])
assert ans_counting_fill([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 3
assert ans_simulate_membership([1, 1, 0, 0], [0, 1, 0, 1]) == 0
assert ans_sandwiches_served([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 3
assert ans_blocking_type([2, 2, 1], [1, 0, 2]) == 0
assert ans_blocking_type([0, 1], [1, 0]) is None
assert ans_min_unable_rearranged([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]) == 1
# kiểm tra Bài 6 bằng vét cạn mọi hoán vị nhỏ
from itertools import permutations
for st, sw in [([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1]), ([0, 0, 1], [1, 1, 1])]:
    best = min(Solution().countStudents(st, list(p)) for p in set(permutations(sw)))
    assert best == ans_min_unable_rearranged(st, sw), (st, sw, best)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countStudents(self, students, sandwiches):
        cnt = [0, 0]
        for s in students:            # thứ tự hàng không quan trọng -> chỉ đếm
            cnt[s] += 1
        for t in sandwiches:          # bánh trên cùng trước (i = 0)
            if cnt[t] == 0:           # không ai muốn -> kẹt, mọi bánh dưới cũng kẹt
                break
            cnt[t] -= 1
        return cnt[0] + cnt[1]
```

**Khung tư duy:** mô phỏng có xoay vòng → hỏi "thứ tự có quan trọng không?" → nếu không, **đếm** → xác định điểm **kẹt** (`break`).

**Checklist trước khi submit**
- [ ] Bánh `sandwiches[0]` là **trên cùng**?
- [ ] Gặp bánh không ai thích thì **dừng** (không bỏ qua)?
- [ ] Nếu mô phỏng: có điều kiện dừng chống lặp vô hạn? Dùng `deque` thay `pop(0)`?
- [ ] Đã thử: kẹt ngay đầu, kẹt ở bánh cuối, phát hết?
- [ ] Tên method `countStudents`, trả về `int`?